## Loading data

In [1]:
!pip install llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [2]:
from llama_index.core import SimpleDirectoryReader

dataset_path = '/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs'
documents = SimpleDirectoryReader(
    dataset_path,
    required_exts=['.pdf']
).load_data()

2026-05-11 23:47:46,670 - INFO - NumExpr defaulting to 4 threads.


In [3]:
print(documents[0].text[:1000])
print(documents[0].metadata)
print("\nTotal pages parsed: ", len(documents))

Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic

{'page_label': '1', 'file_name': '22365_19_Agents_v8.pdf', 'file_path': '/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs/22365_19_Agents_v8.pdf', 'file_type': 'application/pdf', 'file_size': 9305713, 'creation_date': '2026-05-09', 'last_modified_date': '2026-05-09'}

Total pages parsed:  260


Theres also another work around for this that is as follows:

In [4]:
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset_path = Path('/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs')

loader = PDFReader()

all_documents = []

for pdf in dataset_path.glob('*.pdf'):
    doc = loader.load_data(file=pdf)
    all_documents.extend(doc)

print('Total document pages parsed: ', len(all_documents))
print(all_documents[0].metadata)
print(all_documents[0].text[:500])

Total document pages parsed:  260
{'page_label': '1', 'file_name': '22365_19_Agents_v8.pdf'}
Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



Okay so what i noticed here the PDFReader methods returns a list, what it did for each pdf was...
* read each page
* make a seperate entry for each page in the pdf
* the metadata for content of each page consists of its page_label(page number) and the file_name(the file it was extracted from)

This definetly saved the time to write the logic to save each page seperately with its metadata.

## Chunking

In [5]:
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface import HuggingFaceLLM

# Set the default embedding model to a local one sicne we dont have an OpenAI key
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device='cuda'
)

#here this does chunking, creates embedings and stores them in memory
vector_index = VectorStoreIndex.from_documents(documents) 

# Define a free, local LLM 
# We use TinyLlama here because it's small and fits easily in Kaggle memory
Settings.llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.7, "do_sample": False},
    device_map="cuda", # Use that GPU!
)

#converts the embeddigns into a query engine bascically a simple RAG pipeline
query_engine = vector_index.as_query_engine()
response = query_engine.query('What is the main topic of this data?')

2026-05-11 23:48:34.293396: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778543314.500126      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778543314.561738      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778543315.054401      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778543315.054449      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778543315.054452      23 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [7]:
print(response)


The main topic of this data is the development of neural networks for language modeling.
